## Scenario - 1: A single data scientist participating in an ML competition

#### MLflow setup
- Tracking server: No
- Backend store: Local filesystem
- Artifacts store: Local filesystem

#### The experiments can be stored locally by launching MLflow UI

In [8]:
import mlflow

In [9]:
mlflow.get_tracking_uri()

'file:///workspaces/mlops-zoomcamp-megh/02-experiment-tracking/running-mlflow-examples/mlruns'

In [10]:
mlflow.search_experiments()

[<Experiment: artifact_location='file:///workspaces/mlops-zoomcamp-megh/02-experiment-tracking/running-mlflow-examples/mlruns/921674068631269603', creation_time=1780986267130, experiment_id='921674068631269603', last_update_time=1780986267130, lifecycle_stage='active', name='iris-classification', tags={}>,
 <Experiment: artifact_location='file:///workspaces/mlops-zoomcamp-megh/02-experiment-tracking/running-mlflow-examples/mlruns/0', creation_time=1780986080539, experiment_id='0', last_update_time=1780986080539, lifecycle_stage='active', name='Default', tags={}>]

#### Creating new experiment and logging new run

In [14]:
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from mlflow.models.signature import infer_signature

mlflow.set_experiment("iris-classification")

with mlflow.start_run():
    
    mlflow.set_tag("Developer Name", "megh")
    mlflow.set_tag("model", "logistic_regression")
    
    X, y = load_iris(return_X_y=True)
    
    params = {"C": 0.1, "random_state": 42}
    mlflow.log_params(params)
    
    logistic_regression = LogisticRegression(**params).fit(X, y)
    y_pred = logistic_regression.predict(X)
    
    # Infer model signature and set a small input example to avoid the warning
    signature = infer_signature(X, y_pred)
    input_example = X[:3]
    mlflow.sklearn.log_model(logistic_regression, "model", signature=signature, input_example=input_example)
    
    print(f"default artifact location: {mlflow.get_artifact_uri()}")
    mlflow.log_metric("accuracy", accuracy_score(y, y_pred))


2026/06/09 06:38:34 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


2026/06/09 06:38:37 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /tmp/tmp3fjgrae_/model/model.pkl, flavor: sklearn). Fall back to return ['scikit-learn==1.0.2', 'cloudpickle==2.0.0']. Set logging level to DEBUG to see the full traceback. 


default artifact location: file:///workspaces/mlops-zoomcamp-megh/02-experiment-tracking/running-mlflow-examples/mlruns/921674068631269603/5c197aeac4f5485592aa4fdb95e4b627/artifacts


In [16]:
mlflow.search_experiments()

[<Experiment: artifact_location='file:///workspaces/mlops-zoomcamp-megh/02-experiment-tracking/running-mlflow-examples/mlruns/921674068631269603', creation_time=1780986267130, experiment_id='921674068631269603', last_update_time=1780986267130, lifecycle_stage='active', name='iris-classification', tags={}>,
 <Experiment: artifact_location='file:///workspaces/mlops-zoomcamp-megh/02-experiment-tracking/running-mlflow-examples/mlruns/0', creation_time=1780986080539, experiment_id='0', last_update_time=1780986080539, lifecycle_stage='active', name='Default', tags={}>]

#### Interacting with MLflow model registry

In [17]:
from mlflow.tracking import MlflowClient
from mlflow.exceptions import MlflowException

client = MlflowClient()

In [19]:
try:
    client.search_registered_models()
except MlflowException as e:
    print(e)

In [ ]:
# The mlflow model works in this version without a backend store because it is a new version. 
# Before, mlflow used to throw an error when we tried to search for registered models without a backend store.